In [1]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_formula_enrichment = True
pipeline_options.generate_page_images = False
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4, device=AcceleratorDevice.CPU
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

data_dir = "../data"
pdf_files = [f"{data_dir}/1810.04805.pdf", f"{data_dir}/2302.13971.pdf"]
print(f"Will process {len(pdf_files)} PDFs")
for p in pdf_files:
    print(f"  {Path(p).stem}")

/Users/devrahulbanjara/Desktop/production-agentic-rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Will process 2 PDFs
  1810.04805
  2302.13971


In [2]:
import re
from docling_core.types.doc import TextItem, SectionHeaderItem, TableItem, PictureItem

SKIP_SECTIONS = {"Table of Contents", "List of Figures", "List of Tables"}
MIN_CHUNK_CHARS = 20


def infer_heading_level(text: str) -> int:
    """Infer heading level from numbering pattern. Covers arXiv LaTeX conventions."""
    t = text.strip()
    if re.match(r"^\d+\.\d+\.\d+", t):
        return 3
    elif re.match(r"^\d+\.\d+", t):
        return 2
    elif re.match(r"^[A-Z]\.\d+", t):
        return 2
    elif re.match(r"^\d+\s", t):
        return 1
    else:
        return 1


def build_tree(doc):
    """Build structured JSON tree from Docling document."""
    for item, _ in doc.iterate_items():
        if isinstance(item, SectionHeaderItem):
            item.level = infer_heading_level(item.text)

    structured = {"title": doc.name, "sections": []}
    section_stack = []

    for item, _ in doc.iterate_items():
        if isinstance(item, SectionHeaderItem):
            section = {
                "heading": item.text,
                "level": item.level,
                "content": [],
                "children": [],
            }
            while section_stack and section_stack[-1][0] >= item.level:
                section_stack.pop()
            if section_stack:
                section_stack[-1][1]["children"].append(section)
            else:
                structured["sections"].append(section)
            section_stack.append((item.level, section))

        elif isinstance(item, TableItem):
            df = item.export_to_dataframe(doc=doc)
            rows = [df.columns.tolist()] + df.values.tolist()
            caption = item.caption_text(doc=doc)
            entry = {"type": "table", "caption": caption or None, "rows": rows}
            if section_stack:
                section_stack[-1][1]["content"].append(entry)
            else:
                structured.setdefault("preamble", []).append(entry)

        elif isinstance(item, PictureItem):
            caption = item.caption_text(doc=doc)
            entry = {
                "type": "figure",
                "caption": caption or None,
                "has_image": item.image is not None,
            }
            if section_stack:
                section_stack[-1][1]["content"].append(entry)
            else:
                structured.setdefault("preamble", []).append(entry)

        elif isinstance(item, TextItem):
            text = item.text
            if text.strip().startswith("$") or "\\frac" in text or "\\sum" in text:
                entry = {"type": "equation", "latex": text}
            else:
                entry = {"type": "paragraph", "text": text}
            if section_stack:
                section_stack[-1][1]["content"].append(entry)
            else:
                structured.setdefault("preamble", []).append(entry)

    return structured


def is_noise(text: str) -> bool:
    """Filter out Docling artifacts and non-content text."""
    t = text.strip()
    if len(t) < MIN_CHUNK_CHARS:
        return True
    if re.match(r"^DRAFT\b", t):
        return True
    if re.match(r"^Keywords?:", t):
        return True
    if re.match(r"^[\w.\-]+@[\w.\-]+\.\w+$", t):
        return True
    return False


def chunk_document(structured, arxiv_id):
    """Create paragraph chunks with section path prefix from structured tree."""
    chunks = []

    def walk(sections, path):
        for section in sections:
            heading = section["heading"]
            if heading in SKIP_SECTIONS:
                continue
            current_path = path + [heading]

            for item in section["content"]:
                if item["type"] != "paragraph":
                    continue
                text = item["text"].strip()
                if is_noise(text):
                    continue

                section_str = " > ".join(current_path)
                prefixed = f"[Paper: {arxiv_id} | Section: {section_str}]\n{text}"
                chunks.append(
                    {
                        "text": prefixed,
                        "arxiv_id": arxiv_id,
                        "chunk_type": "paragraph",
                        "section_path": current_path,
                    }
                )

            walk(section["children"], current_path)

    walk(structured["sections"], [])
    return chunks


print("Helper functions defined")

Helper functions defined


In [3]:
all_chunks = []
all_trees = {}

for pdf_path in pdf_files:
    arxiv_id = Path(pdf_path).stem
    print(f"Processing {arxiv_id}...")

    result = converter.convert(pdf_path)
    tree = build_tree(result.document)
    all_trees[arxiv_id] = tree

    chunks = chunk_document(tree, arxiv_id)
    all_chunks.extend(chunks)
    print(f"  → {len(tree['sections'])} sections, {len(chunks)} chunks")

print(f"\nTotal: {len(pdf_files)} papers, {len(all_chunks)} chunks")

Processing 1810.04805...


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 7876.76it/s]
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 5288.40it/s]
[transformers] The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  → 13 sections, 210 chunks
Processing 2302.13971...


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  → 39 sections, 273 chunks

Total: 2 papers, 483 chunks


In [4]:
from collections import Counter

papers = Counter(c["arxiv_id"] for c in all_chunks)
print("Chunks per paper:")
for paper, count in papers.most_common():
    print(f"  {paper}: {count}")

avg_len = sum(len(c["text"]) for c in all_chunks) / len(all_chunks)
print(f"\nAvg chunk length: {avg_len:.0f} chars")
print(f"\nSample chunk:\n{'─' * 60}")
print(all_chunks[0]["text"][:400])

Chunks per paper:
  2302.13971: 273
  1810.04805: 210

Avg chunk length: 343 chars

Sample chunk:
────────────────────────────────────────────────────────────
[Paper: 1810.04805 | Section: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding]
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova


In [5]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import PointStruct
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

collection_name = "arxiv_papers_docling"
dense_model_name = "BAAI/bge-small-en"
sparse_model_name = "qdrant/bm25"

client = QdrantClient(url="http://localhost:6333")

# Fresh collection
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense_vector": models.VectorParams(size=384, distance=models.Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25_sparse_vector": models.SparseVectorParams(modifier=models.Modifier.IDF)
    },
)

# Embed all chunks
texts = [c["text"] for c in all_chunks]

dense_encoder = SentenceTransformer(dense_model_name)
dense_embeddings = dense_encoder.encode(texts, show_progress_bar=True)

bm25_encoder = SparseTextEmbedding(model_name=sparse_model_name)
sparse_embeddings = list(bm25_encoder.embed(texts))

print(f"Dense shape: {dense_embeddings.shape}")
print(f"Sparse vectors: {len(sparse_embeddings)}")

# Upload to Qdrant
points = []
for idx, (chunk, dense_vec, sparse_vec) in enumerate(
    zip(all_chunks, dense_embeddings, sparse_embeddings)
):
    point = PointStruct(
        id=idx + 1,
        payload={
            "text": chunk["text"],
            "arxiv_id": chunk["arxiv_id"],
            "chunk_type": chunk["chunk_type"],
            "section_path": chunk["section_path"],
        },
        vector={
            "dense_vector": dense_vec.tolist(),
            "bm25_sparse_vector": models.SparseVector(
                indices=sparse_vec.indices.tolist(),
                values=sparse_vec.values.tolist(),
            ),
        },
    )
    points.append(point)

client.upload_points(collection_name=collection_name, points=points, batch_size=64)
print(f"\nUploaded {len(points)} chunks to '{collection_name}'")

Batches: 100%|██████████| 16/16 [00:06<00:00,  2.32it/s]


Dense shape: (483, 384)
Sparse vectors: 483

Uploaded 483 chunks to 'arxiv_papers_docling'


In [10]:
from pprint import pprint

query = "How many GPU hours did it take to train LLaMA-65B?"

dense_query_vec = dense_encoder.encode(query)
sparse_query_vec = next(bm25_encoder.embed([query]))

In [11]:
results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_query_vec.tolist(), using="dense_vector", limit=5),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using="bm25_sparse_vector",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)

pprint(results.points)

[ScoredPoint(id=246, version=4, score=0.8333334, payload={'text': '[Paper: 2302.13971 | Section: 2 Approach > 2.4 Efficient implementation]\nWhen training a 65B-parameter model, our code processes around 380 tokens/sec/GPU on 2048 A100 GPU with 80GB of RAM. This means that training over our dataset containing 1.4T tokens takes approximately 21 days.', 'arxiv_id': '2302.13971', 'chunk_type': 'paragraph', 'section_path': ['2 Approach', '2.4 Efficient implementation']}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=213, version=4, score=0.5833334, payload={'text': '[Paper: 2302.13971 | Section: LLaMA: Open and Efficient Foundation Language Models]\nThe focus of this work is to train a series of language models that achieve the best possible performance at various inference budgets, by training on more tokens than what is typically used. The resulting models, called LLaMA , ranges from 7B to 65B parameters with competitive performance compared to the best existing LLMs. F